In [ ]:
# Load datasets via DataLoader
import sys
sys.path.append('../src')
from data_loader import DataLoader
from preprocessing import FraudDataPreprocessor
import matplotlib.pyplot as plt
import seaborn as sns

dl = DataLoader(data_dir='../data/raw')
fraud_df = dl.load_fraud_data('Fraud_Data.csv')
ip_df = dl.load_ip_country_data('IpAddress_to_Country.csv')
credit_df = dl.load_creditcard_data('creditcard.csv')

In [ ]:
print('duplicates in lower_bound_ip_address:', ip_df['lower_bound_ip_address'].duplicated().sum())
print('duplicates in upper_bound_ip_address:', ip_df['upper_bound_ip_address'].duplicated().sum())

In [ ]:
# Basic info and cleaning using the preprocessor
prep = FraudDataPreprocessor()
fraud_clean = prep.clean_data(fraud_df)
fraud_features = prep.create_time_features(fraud_clean)

# Attempt IP->country merge using robust helper (handles dtype coercion and dropna steps)
merged = None
try:
    merged = prep.merge_with_ip_data(fraud_features, ip_df)
    print('Merged (with country) rows:', len(merged))
    display(merged.head())
except Exception as e:
    print('Initial IP merge failed:', e)
    # Try coercing IP columns to strings and retry
    if 'ip_address' in fraud_features.columns:
        fraud_features['ip_address'] = fraud_features['ip_address'].astype(str)
    if 'lower_bound_ip_address' in ip_df.columns:
        ip_df['lower_bound_ip_address'] = ip_df['lower_bound_ip_address'].astype(str)
    if 'upper_bound_ip_address' in ip_df.columns:
        ip_df['upper_bound_ip_address'] = ip_df['upper_bound_ip_address'].astype(str)
    try:
        merged = prep.merge_with_ip_data(fraud_features, ip_df)
        print('Merged after dtype coercion:', len(merged))
        display(merged.head())
    except Exception as e2:
        print('Merge still failed:', e2)
        # Fall back to unmerged dataset with a warning
        merged = fraud_features.copy()
        print('Proceeding without country enrichment (some analyses will be limited).')

print('Cleaned fraud rows:', len(fraud_clean))
print('Merged (with country) rows (final):', len(merged))

In [ ]:
# Expanded EDA: class distribution, univariate, bivariate, country analysis, and train-only SMOTE demonstration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
import os
from sklearn.model_selection import train_test_split
from collections import Counter

# Ensure we have the relevant time-based features
if 'purchase_time' in merged.columns:
    merged['purchase_time'] = pd.to_datetime(merged['purchase_time'])
    if 'purchase_hour' not in merged.columns:
        merged['purchase_hour'] = merged['purchase_time'].dt.hour
    if 'day_of_week' not in merged.columns:
        merged['day_of_week'] = merged['purchase_time'].dt.dayofweek

# Class distribution (counts and percentages)
if 'class' in merged.columns:
    counts = merged['class'].value_counts().sort_index()
    pct = merged['class'].value_counts(normalize=True).sort_index() * 100
    print('Class counts:\n', counts)
    print('\nClass distribution (%):\n', pct.round(2))

    plt.figure(figsize=(6,4))
    sns.countplot(x='class', data=merged)
    plt.title('Fraud (1) vs Legit (0)')
    plt.savefig('../reports/figures/class_distribution_all.png')
    plt.show()

# Purchase value distribution by class
if 'purchase_value' in merged.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x='class', y='purchase_value', data=merged)
    plt.title('Purchase Value by Class')
    plt.yscale('log')  # if skewed
    plt.savefig('../reports/figures/purchase_value_by_class.png')
    plt.show()

# Fraud rate by country
if 'country' in merged.columns:
    country_stats = merged.groupby('country')['class'].agg(['count','mean']).rename(columns={'mean':'fraud_rate'})
    country_stats = country_stats.sort_values('fraud_rate', ascending=False)
    display(country_stats.head(10))

    plt.figure(figsize=(10,5))
    sns.barplot(x=country_stats.head(10).index, y=country_stats.head(10)['fraud_rate'])
    plt.xticks(rotation=45)
    plt.title('Top 10 Countries by Fraud Rate')
    plt.ylabel('Fraud rate (mean of class)')
    plt.savefig('../reports/figures/fraud_by_country.png')
    plt.show()
else:
    print('No country column present; IP merge may have failed earlier.')

# Hour-of-day fraud rate
if 'purchase_hour' in merged.columns:
    hour_stats = merged.groupby('purchase_hour')['class'].agg(['count','mean']).rename(columns={'mean':'fraud_rate'})
    plt.figure(figsize=(10,4))
    sns.lineplot(x=hour_stats.index, y=hour_stats['fraud_rate'], marker='o')
    plt.title('Fraud rate by Hour of Day')
    plt.xlabel('Hour of day')
    plt.ylabel('Fraud rate')
    plt.savefig('../reports/figures/fraud_rate_by_hour.png')
    plt.show()

# Train/test split and SMOTE on training data only (demonstration)
if 'class' in merged.columns:
    X = merged.drop(columns=['class'])
    y = merged['class']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    print('Training set class distribution before resampling:', Counter(y_train))

    try:
        from imblearn.over_sampling import SMOTE
        sm = SMOTE(random_state=42)
        # SMOTE works on numeric arrays; select numerical features and impute if necessary
        X_train_num = X_train.select_dtypes(include=[np.number]).fillna(0)
        X_res, y_res = sm.fit_resample(X_train_num, y_train)
        print('Training set class distribution after SMOTE:', Counter(y_res))

        # Save before/after plots
        plt.figure()
        pd.Series(y_train).value_counts().sort_index().plot(kind='bar')
        plt.title('Train class distribution BEFORE SMOTE')
        plt.savefig('../reports/figures/class_distribution_before.png')
        plt.close()

        plt.figure()
        pd.Series(y_res).value_counts().sort_index().plot(kind='bar')
        plt.title('Train class distribution AFTER SMOTE')
        plt.savefig('../reports/figures/class_distribution_after_smote.png')
        plt.close()

    except Exception as e:
        print('SMOTE could not be applied (missing package or other issue):', e)
        # Fallback: use undersampling (RandomUnderSampler) to demonstrate an alternative
        try:
            from imblearn.under_sampling import RandomUnderSampler
            rus = RandomUnderSampler(random_state=42)
            X_train_num = X_train.select_dtypes(include=[np.number]).fillna(0)
            X_rus, y_rus = rus.fit_resample(X_train_num, y_train)
            print('Training set class distribution after RandomUnderSampler (fallback):', Counter(y_rus))

            # Save before/after plots for undersampling
            plt.figure()
            pd.Series(y_train).value_counts().sort_index().plot(kind='bar')
            plt.title('Train class distribution BEFORE undersampling')
            plt.savefig('../reports/figures/class_distribution_before.png')
            plt.close()

            plt.figure()
            pd.Series(y_rus).value_counts().sort_index().plot(kind='bar')
            plt.title('Train class distribution AFTER undersampling')
            plt.savefig('../reports/figures/class_distribution_after_undersample.png')
            plt.close()

        except Exception as e2:
            print('Undersampling fallback also failed:', e2)
            print('No resampling applied. Documented that resampling could not be performed in this environment.')

# Display generated figures if available
fig_dir = '../reports/figures'
for fname in ['fraud_by_country.png','class_distribution_all.png','class_distribution_before.png','class_distribution_after_smote.png','fraud_rate_by_hour.png']:
    path = os.path.join(fig_dir, fname)
    if os.path.exists(path):
        print(f'Displaying {fname}')
        display(Image(filename=path))
    else:
        print(f'Figure not found: {fname} (path: {path})')

In [ ]:
# Save processed fraud data for downstream notebooks
out_path = '../data/processed/fraud_data_processed.csv'
merged.to_csv(out_path, index=False)
print(f'Saved processed fraud data to: {out_path}')

# EDA Summary & Next Steps

**Findings:**

- IP-to-country enrichment is attempted and applied where IP ranges match; see counts above and `reports/figures/fraud_by_country.png` if present. If the IP merge failed for some rows, the notebook proceeds without country for those rows and reports the final merged row counts.
- The dataset is imbalanced (fraud is the minority class). We demonstrate SMOTE applied to the **training set only**, and save before/after plots to `reports/figures/class_distribution_before.png` and `reports/figures/class_distribution_after_smote.png` where applicable. A resampling template and a robust train-only SMOTE example are available in `notebooks/feature-engineering.ipynb`. The transformed features for modeling are saved to `data/processed/fraud_data_features_transformed.csv`.
- Key predictive signals identified: transaction amount (`purchase_value`), time features (`purchase_hour`, `day_of_week`), country (where available), and short-window transaction burst features (`txn_count_24h`, `avg_time_between_txn_hours`).

**Next steps:**

1. Proceed to hyperparameter tuning and model selection (recommend using Optuna with stratified/time-aware CV).
2. Persist and review SMOTE-augmented training splits and evaluate on untouched test sets to avoid leakage.
3. Generate final figures (feature importance, SHAP summary) and include them in the final report (`reports/figures/`).

If you'd like, I can run the hyperparameter tuning and create the final evaluation & explainability figures next.